In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, lit, current_timestamp, to_json, struct,
    year, month, quarter, weekofyear, dayofmonth, date_format,
    dayofweek, when, explode, sequence
)
import uuid

# =========================================================
# CONFIG
# =========================================================
pipeline_name = "retail_pipeline"
layer_name = "gold"
load_type = "FULL"
triggered_by = "manual"
source_system = "Postgres"

audit_table = "retail.audit.etl_run_log"
reject_table = "retail.audit.etl_reject_log"

batch_id = "batch_gold_full_20260423"

# Source silver tables
silver_brand = "retail.silver.ns_brands"
silver_customer = "retail.silver.ns_customers"
silver_item = "retail.silver.ns_items"
silver_sales_orders = "retail.silver.ns_sales_orders"
silver_sales_order_lines = "retail.silver.ns_sales_order_lines"
silver_shipments = "retail.silver.ns_shipments"

# Target gold tables
gold_dim_brand = "retail.gold.dim_brand"
gold_dim_customer = "retail.gold.dim_customer"
gold_dim_item = "retail.gold.dim_item"
gold_dim_date = "retail.gold.dim_date"
gold_fact_sales = "retail.gold.fact_sales"

# =========================================================
# HELPERS
# =========================================================
def generate_run_id():
    return str(uuid.uuid4())

def start_audit(run_id, table_name):
    spark.sql(f"""
        INSERT INTO {audit_table}
        (
          run_id, batch_id, pipeline_name, layer_name, table_name, source_system,
          load_type, triggered_by, start_time, end_time, duration_seconds,
          status, rows_read, rows_written, rows_rejected, records_before,
          records_after, error_message, created_at
        )
        VALUES (
          '{run_id}', '{batch_id}', '{pipeline_name}', '{layer_name}', '{table_name}', '{source_system}',
          '{load_type}', '{triggered_by}', current_timestamp(), NULL, NULL,
          'STARTED', NULL, NULL, NULL, NULL,
          NULL, NULL, current_timestamp()
        )
    """)

def end_audit_success(run_id, rows_read, rows_written, rows_rejected, records_before, records_after):
    spark.sql(f"""
        UPDATE {audit_table}
        SET
          end_time = current_timestamp(),
          duration_seconds = unix_timestamp(current_timestamp()) - unix_timestamp(start_time),
          status = 'SUCCESS',
          rows_read = {rows_read},
          rows_written = {rows_written},
          rows_rejected = {rows_rejected},
          records_before = {records_before},
          records_after = {records_after}
        WHERE run_id = '{run_id}'
    """)

def end_audit_fail(run_id, error_message):
    safe_error = error_message.replace("'", "''")[:5000]
    spark.sql(f"""
        UPDATE {audit_table}
        SET
          end_time = current_timestamp(),
          duration_seconds = unix_timestamp(current_timestamp()) - unix_timestamp(start_time),
          status = 'FAILED',
          error_message = '{safe_error}'
        WHERE run_id = '{run_id}'
    """)

def log_rejects(df_rejects, run_id, table_name):
    if df_rejects.isEmpty():
        return

    (
        df_rejects
        .withColumn("reject_id", F.expr("uuid()"))
        .withColumn("run_id", lit(run_id))
        .withColumn("batch_id", lit(batch_id))
        .withColumn("pipeline_name", lit(pipeline_name))
        .withColumn("layer_name", lit(layer_name))
        .withColumn("table_name", lit(table_name))
        .withColumn("source_system", lit(source_system))
        .withColumn("rejected_at", current_timestamp())
        .withColumn("created_at", current_timestamp())
        .select(
            "reject_id", "run_id", "batch_id", "pipeline_name", "layer_name",
            "table_name", "source_system", "source_record_id", "reject_stage",
            "reject_reason", "reject_column", "severity", "raw_payload",
            "error_code", "error_message", "rejected_at", "created_at"
        )
        .write.format("delta").mode("append").saveAsTable(reject_table)
    )

# =========================================================
# DIM BRAND
# =========================================================
def load_dim_brand():
    table_name = "dim_brand"
    run_id = generate_run_id()
    start_audit(run_id, table_name)

    try:
        df_src = spark.table(silver_brand)
        rows_read = df_src.count()
        records_before = spark.table(gold_dim_brand).count()

        df_tgt = (
            df_src
            .select(
                col("internal_id").alias("brand_internal_id"),
                "brand_code",
                "brand_name",
                "is_active",
                "source_system",
                "etl_run_id",
                "record_hash",
                "created_at",
                "etl_loaded_at"
            )
            .dropDuplicates(["brand_internal_id"])
        )

        rows_written = df_tgt.count()

        spark.sql(f"TRUNCATE TABLE {gold_dim_brand}")

        df_tgt.write \
            .mode("append") \
            .format("delta") \
            .saveAsTable(gold_dim_brand)

        records_after = spark.table(gold_dim_brand).count()

        end_audit_success(run_id, rows_read, rows_written, 0, records_before, records_after)
        print(f"{gold_dim_brand}: {rows_written} rows loaded successfully")

    except Exception as e:
        end_audit_fail(run_id, str(e))
        raise

# =========================================================
# DIM CUSTOMER
# =========================================================
def load_dim_customer():
    table_name = "dim_customer"
    run_id = generate_run_id()
    start_audit(run_id, table_name)

    try:
        df_src = spark.table(silver_customer)
        rows_read = df_src.count()
        records_before = spark.table(gold_dim_customer).count()

        df_tgt = (
            df_src
            .select(
                col("internal_id").alias("customer_internal_id"),
                "entity_id",
                "company_name",
                "customer_type",
                "email",
                "phone",
                "subsidiary",
                "currency_code",
                "terms_name",
                "vat_number",
                "billing_address1",
                "billing_city",
                "billing_state",
                "billing_country",
                "shipping_address1",
                "shipping_city",
                "shipping_state",
                "shipping_country",
                "is_active",
                "source_system",
                "etl_run_id",
                "record_hash",
                "created_at",
                "etl_loaded_at"
            )
            .dropDuplicates(["customer_internal_id"])
        )

        rows_written = df_tgt.count()

        spark.sql(f"TRUNCATE TABLE {gold_dim_customer}")

        df_tgt.write \
            .mode("append") \
            .format("delta") \
            .saveAsTable(gold_dim_customer)

        records_after = spark.table(gold_dim_customer).count()

        end_audit_success(run_id, rows_read, rows_written, 0, records_before, records_after)
        print(f"{gold_dim_customer}: {rows_written} rows loaded successfully")

    except Exception as e:
        end_audit_fail(run_id, str(e))
        raise

# =========================================================
# DIM ITEM
# =========================================================
def load_dim_item():
    table_name = "dim_item"
    run_id = generate_run_id()
    start_audit(run_id, table_name)

    try:
        df_src = spark.table(silver_item)
        rows_read = df_src.count()
        records_before = spark.table(gold_dim_item).count()

        df_tgt = (
            df_src
            .select(
                col("internal_id").alias("item_internal_id"),
                "item_id",
                "item_name",
                "item_type",
                "category",
                "subcategory",
                "brand_internal_id",
                "base_price",
                "cost_price",
                "is_active",
                "source_system",
                "etl_run_id",
                "record_hash",
                "created_at",
                "etl_loaded_at"
            )
            .dropDuplicates(["item_internal_id"])
        )

        rows_written = df_tgt.count()

        spark.sql(f"TRUNCATE TABLE {gold_dim_item}")

        df_tgt.write \
            .mode("append") \
            .format("delta") \
            .saveAsTable(gold_dim_item)

        records_after = spark.table(gold_dim_item).count()

        end_audit_success(run_id, rows_read, rows_written, 0, records_before, records_after)
        print(f"{gold_dim_item}: {rows_written} rows loaded successfully")

    except Exception as e:
        end_audit_fail(run_id, str(e))
        raise

# =========================================================
# DIM DATE
# =========================================================
def load_dim_date():
    table_name = "dim_date"
    run_id = generate_run_id()
    start_audit(run_id, table_name)

    try:
        df_orders = spark.table(silver_sales_orders).select(col("order_date").alias("dt"))
        df_shipments_1 = spark.table(silver_shipments).select(col("shipment_date").alias("dt"))
        df_shipments_2 = spark.table(silver_shipments).select(col("delivery_date").alias("dt"))

        df_dates = (
            df_orders.union(df_shipments_1).union(df_shipments_2)
            .filter(col("dt").isNotNull())
        )

        rows_read = df_dates.count()
        records_before = spark.table(gold_dim_date).count()

        min_max = df_dates.agg(
            F.min("dt").alias("min_date"),
            F.max("dt").alias("max_date")
        ).collect()[0]

        min_date = min_max["min_date"]
        max_date = min_max["max_date"]

        df_calendar = (
            spark.createDataFrame([(min_date, max_date)], ["start_date", "end_date"])
            .select(explode(sequence(col("start_date"), col("end_date"))).alias("full_date"))
        )

        df_tgt = (
            df_calendar
            .select(
                date_format(col("full_date"), "yyyyMMdd").cast("int").alias("date_key"),
                col("full_date"),
                dayofmonth(col("full_date")).alias("day_of_month"),
                month(col("full_date")).alias("month_num"),
                date_format(col("full_date"), "MMMM").alias("month_name"),
                quarter(col("full_date")).alias("quarter_num"),
                year(col("full_date")).alias("year_num"),
                weekofyear(col("full_date")).alias("week_of_year"),
                dayofweek(col("full_date")).alias("day_of_week_num"),
                date_format(col("full_date"), "EEEE").alias("day_of_week_name"),
                when(dayofweek(col("full_date")).isin(1, 7), True).otherwise(False).alias("is_weekend")
            )
        )

        rows_written = df_tgt.count()

        spark.sql(f"TRUNCATE TABLE {gold_dim_date}")

        df_tgt.write \
            .mode("append") \
            .format("delta") \
            .saveAsTable(gold_dim_date)

        records_after = spark.table(gold_dim_date).count()

        end_audit_success(run_id, rows_read, rows_written, 0, records_before, records_after)
        print(f"{gold_dim_date}: {rows_written} rows loaded successfully")

    except Exception as e:
        end_audit_fail(run_id, str(e))
        raise

# =========================================================
# RUN ALL
# =========================================================
load_dim_brand()
load_dim_customer()
load_dim_item()
load_dim_date()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, current_timestamp, to_json, struct

# =========================================================
# FACT SALES ONLY
# =========================================================
def load_fact_sales():
    table_name = "fact_sales"
    run_id = generate_run_id()
    start_audit(run_id, table_name)

    try:
        records_before = spark.table(gold_fact_sales).count()

        # -------------------------------------------------
        # 1. Load and rename source tables explicitly
        # -------------------------------------------------
        df_sol = (
            spark.table(silver_sales_order_lines)
            .select(
                col("internal_id").alias("sol_internal_id"),
                col("sales_order_internal_id").alias("sol_sales_order_internal_id"),
                col("line_num").alias("sol_line_num"),
                col("item_internal_id").alias("sol_item_internal_id"),
                col("quantity").alias("sol_quantity"),
                col("unit_rate").alias("sol_unit_rate"),
                col("line_amount").alias("sol_line_amount"),
                col("record_hash").alias("sol_record_hash"),
                col("created_at").alias("sol_created_at")
            )
        )

        df_so = (
            spark.table(silver_sales_orders)
            .select(
                col("internal_id").alias("so_internal_id"),
                col("customer_internal_id").alias("so_customer_internal_id"),
                col("order_date").alias("so_order_date"),
                col("order_status").alias("so_order_status"),
                col("sales_channel").alias("so_sales_channel"),
                col("payment_status").alias("so_payment_status"),
                col("currency_code").alias("so_currency_code"),
                col("integration_source").alias("so_integration_source"),
                col("order_total").alias("so_order_total"),
                col("source_system").alias("so_source_system"),
                col("etl_run_id").alias("so_etl_run_id"),
                col("record_hash").alias("so_record_hash")
            )
        )

        df_sh = (
            spark.table(silver_shipments)
            .select(
                col("internal_id").alias("sh_internal_id"),
                col("sales_order_internal_id").alias("sh_sales_order_internal_id"),
                col("shipment_date").alias("sh_shipment_date"),
                col("delivery_date").alias("sh_delivery_date"),
                col("shipment_status").alias("sh_shipment_status"),
                col("warehouse_name").alias("sh_warehouse_name"),
                col("tracking_number").alias("sh_tracking_number"),
                col("record_hash").alias("sh_record_hash")
            )
        )

        # -------------------------------------------------
        # 2. Join transactional tables
        # -------------------------------------------------
        df_joined = (
            df_sol
            .join(
                df_so,
                col("sol_sales_order_internal_id") == col("so_internal_id"),
                "inner"
            )
            .join(
                df_sh,
                col("so_internal_id") == col("sh_sales_order_internal_id"),
                "left"
            )
        )

        rows_read = df_joined.count()

        # -------------------------------------------------
        # 3. Lookup dimensions with renamed lookup columns
        # -------------------------------------------------
        df_customer_lkp = spark.table(gold_dim_customer).select(
            col("customer_key"),
            col("customer_internal_id").alias("lkp_customer_internal_id")
        )

        df_item_lkp = spark.table(gold_dim_item).select(
            col("item_key"),
            col("item_internal_id").alias("lkp_item_internal_id")
        )

        df_date_lkp = spark.table(gold_dim_date).select(
            col("date_key"),
            col("full_date").alias("lkp_full_date")
        )

        df_fact_pre = (
            df_joined
            .join(
                df_customer_lkp,
                col("so_customer_internal_id") == col("lkp_customer_internal_id"),
                "left"
            )
            .join(
                df_item_lkp,
                col("sol_item_internal_id") == col("lkp_item_internal_id"),
                "left"
            )
            .join(
                df_date_lkp,
                col("so_order_date") == col("lkp_full_date"),
                "left"
            )
        )

        # -------------------------------------------------
        # 4. Reject dimension lookup failures
        # -------------------------------------------------
        reject_condition = (
            col("customer_key").isNull() |
            col("item_key").isNull() |
            col("date_key").isNull()
        )

        df_rejects = (
            df_fact_pre
            .filter(reject_condition)
            .withColumn("source_record_id", col("sol_internal_id").cast("string"))
            .withColumn("reject_stage", lit("GOLD"))
            .withColumn("reject_reason", lit("DIMENSION_LOOKUP_FAILED"))
            .withColumn("reject_column", lit("customer_key,item_key,order_date_key"))
            .withColumn("severity", lit("ERROR"))
            .withColumn("raw_payload", to_json(struct(*[col(c) for c in df_fact_pre.columns])))
            .withColumn("error_code", lit("GLD_001"))
            .withColumn("error_message", lit("One or more dimension lookups failed"))
        )

        rows_rejected = df_rejects.count()
        log_rejects(df_rejects, run_id, table_name)

        df_valid = df_fact_pre.filter(~reject_condition)

        # -------------------------------------------------
        # 5. Build fact output
        # -------------------------------------------------
        df_tgt = (
            df_valid
            .select(
                col("sol_internal_id").alias("sales_order_line_internal_id"),
                col("sol_sales_order_internal_id").alias("sales_order_internal_id"),
                col("sol_line_num").alias("sales_order_line_num"),
                col("date_key").alias("order_date_key"),
                col("customer_key"),
                col("item_key"),
                col("sol_quantity").alias("quantity"),
                col("sol_unit_rate").alias("unit_rate"),
                col("sol_line_amount").alias("line_amount"),
                col("so_order_total").alias("order_total"),
                col("so_order_status").alias("order_status"),
                col("so_sales_channel").alias("sales_channel"),
                col("so_payment_status").alias("payment_status"),
                col("so_currency_code").alias("currency_code"),
                col("so_integration_source").alias("integration_source"),
                col("sh_shipment_date").alias("shipment_date"),
                col("sh_delivery_date").alias("delivery_date"),
                col("sh_shipment_status").alias("shipment_status"),
                col("sh_warehouse_name").alias("warehouse_name"),
                col("sh_tracking_number").alias("tracking_number"),
                col("so_source_system").alias("source_system"),
                col("so_etl_run_id").alias("etl_run_id"),
                F.sha2(
                    F.concat_ws(
                        "||",
                        F.coalesce(col("sol_record_hash"), lit("")),
                        F.coalesce(col("so_record_hash"), lit("")),
                        F.coalesce(col("sh_record_hash"), lit(""))
                    ),
                    256
                ).alias("record_hash"),
                col("sol_created_at").alias("created_at"),
                current_timestamp().alias("etl_loaded_at")
            )
            .dropDuplicates(["sales_order_line_internal_id"])
        )

        rows_written = df_tgt.count()

        spark.sql(f"TRUNCATE TABLE {gold_fact_sales}")

        df_tgt.write \
            .mode("append") \
            .format("delta") \
            .saveAsTable(gold_fact_sales)

        records_after = spark.table(gold_fact_sales).count()

        end_audit_success(
            run_id=run_id,
            rows_read=rows_read,
            rows_written=rows_written,
            rows_rejected=rows_rejected,
            records_before=records_before,
            records_after=records_after
        )

        print(f"{gold_fact_sales}: {rows_written} rows loaded successfully")

    except Exception as e:
        end_audit_fail(run_id, str(e))
        raise

# Debugging Fact table load

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

df_sol = (
    spark.table("retail.silver.ns_sales_order_lines")
    .select(
        col("internal_id").alias("sol_internal_id"),
        col("sales_order_internal_id").alias("sol_sales_order_internal_id"),
        col("line_num").alias("sol_line_num"),
        col("item_internal_id").alias("sol_item_internal_id"),
        col("quantity").alias("sol_quantity"),
        col("unit_rate").alias("sol_unit_rate"),
        col("line_amount").alias("sol_line_amount"),
        col("record_hash").alias("sol_record_hash"),
        col("created_at").alias("sol_created_at")
    )
)

df_so = (
    spark.table("retail.silver.ns_sales_orders")
    .select(
        col("internal_id").alias("so_internal_id"),
        col("customer_internal_id").alias("so_customer_internal_id"),
        col("order_date").alias("so_order_date"),
        col("order_status").alias("so_order_status"),
        col("sales_channel").alias("so_sales_channel"),
        col("payment_status").alias("so_payment_status"),
        col("currency_code").alias("so_currency_code"),
        col("integration_source").alias("so_integration_source"),
        col("order_total").alias("so_order_total"),
        col("source_system").alias("so_source_system"),
        col("etl_run_id").alias("so_etl_run_id"),
        col("record_hash").alias("so_record_hash")
    )
)

df_sh = (
    spark.table("retail.silver.ns_shipments")
    .select(
        col("internal_id").alias("sh_internal_id"),
        col("sales_order_internal_id").alias("sh_sales_order_internal_id"),
        col("shipment_date").alias("sh_shipment_date"),
        col("delivery_date").alias("sh_delivery_date"),
        col("shipment_status").alias("sh_shipment_status"),
        col("warehouse_name").alias("sh_warehouse_name"),
        col("tracking_number").alias("sh_tracking_number"),
        col("record_hash").alias("sh_record_hash")
    )
)

df_joined = (
    df_sol
    .join(df_so, col("sol_sales_order_internal_id") == col("so_internal_id"), "inner")
    .join(df_sh, col("so_internal_id") == col("sh_sales_order_internal_id"), "left")
)

print("df_joined =", df_joined.count())

df_customer_lkp = spark.table("retail.gold.dim_customer").select(
    col("customer_key"),
    col("customer_internal_id").cast("int").alias("lkp_customer_internal_id")
)

df_item_lkp = spark.table("retail.gold.dim_item").select(
    col("item_key"),
    col("item_internal_id").cast("int").alias("lkp_item_internal_id")
)

df_date_lkp = spark.table("retail.gold.dim_date").select(
    col("date_key"),
    col("full_date").alias("lkp_full_date")
)

df_fact_pre = (
    df_joined
    .join(
        df_customer_lkp,
        col("so_customer_internal_id").cast("int") == col("lkp_customer_internal_id"),
        "left"
    )
    .join(
        df_item_lkp,
        col("sol_item_internal_id").cast("int") == col("lkp_item_internal_id"),
        "left"
    )
    .join(
        df_date_lkp,
        F.to_date(col("so_order_date")) == col("lkp_full_date"),
        "left"
    )
)

print("missing customer_key =", df_fact_pre.filter(col("customer_key").isNull()).count())
print("missing item_key     =", df_fact_pre.filter(col("item_key").isNull()).count())
print("missing date_key     =", df_fact_pre.filter(col("date_key").isNull()).count())
print("fully matched rows   =", df_fact_pre.filter(
    col("customer_key").isNotNull() &
    col("item_key").isNotNull() &
    col("date_key").isNotNull()
).count())

In [0]:
from pyspark.sql.functions import col

reject_condition = (
    col("customer_key").isNull() |
    col("item_key").isNull() |
    col("date_key").isNull()
)

df_valid = df_fact_pre.filter(~reject_condition)

print("df_valid:", df_valid.count())

df_tgt_preview = df_valid.select(
    col("sol_internal_id").alias("sales_order_line_internal_id")
)

print("df_tgt BEFORE dedupe:", df_tgt_preview.count())

In [0]:
from pyspark.sql.functions import col, countDistinct

df_key_check = df_valid.select(
    col("sol_internal_id").alias("sales_order_line_internal_id")
)

print("total rows          =", df_key_check.count())
print("distinct key rows   =", df_key_check.select(countDistinct("sales_order_line_internal_id")).collect()[0][0])

In [0]:
df_tgt = (
    df_valid
    .select(
        col("sol_internal_id").alias("sales_order_line_internal_id"),
        col("sol_sales_order_internal_id").alias("sales_order_internal_id"),
        col("sol_line_num").alias("sales_order_line_num"),
        col("date_key").alias("order_date_key"),
        col("customer_key"),
        col("item_key"),
        col("sol_quantity").alias("quantity"),
        col("sol_unit_rate").alias("unit_rate"),
        col("sol_line_amount").alias("line_amount"),
        col("so_order_total").alias("order_total"),
        col("so_order_status").alias("order_status"),
        col("so_sales_channel").alias("sales_channel"),
        col("so_payment_status").alias("payment_status"),
        col("so_currency_code").alias("currency_code"),
        col("so_integration_source").alias("integration_source"),
        col("sh_shipment_date").alias("shipment_date"),
        col("sh_delivery_date").alias("delivery_date"),
        col("sh_shipment_status").alias("shipment_status"),
        col("sh_warehouse_name").alias("warehouse_name"),
        col("sh_tracking_number").alias("tracking_number"),
        col("so_source_system").alias("source_system"),
        col("so_etl_run_id").alias("etl_run_id"),
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(col("sol_record_hash"), F.lit("")),
                F.coalesce(col("so_record_hash"), F.lit("")),
                F.coalesce(col("sh_record_hash"), F.lit(""))
            ),
            256
        ).alias("record_hash"),
        col("sol_created_at").alias("created_at"),
        F.current_timestamp().alias("etl_loaded_at")
    )
)

print("df_tgt count =", df_tgt.count())
df_tgt.printSchema()

In [0]:
def load_fact_sales():
    table_name = "fact_sales"
    run_id = generate_run_id()
    start_audit(run_id, table_name)

    try:
        records_before = spark.table(gold_fact_sales).count()

        # -------------------------------------------------
        # 1. Load and rename source tables explicitly
        # -------------------------------------------------
        df_sol = (
            spark.table(silver_sales_order_lines)
            .select(
                col("internal_id").alias("sol_internal_id"),
                col("sales_order_internal_id").alias("sol_sales_order_internal_id"),
                col("line_num").alias("sol_line_num"),
                col("item_internal_id").alias("sol_item_internal_id"),
                col("quantity").alias("sol_quantity"),
                col("unit_rate").alias("sol_unit_rate"),
                col("line_amount").alias("sol_line_amount"),
                col("record_hash").alias("sol_record_hash"),
                col("created_at").alias("sol_created_at")
            )
        )

        df_so = (
            spark.table(silver_sales_orders)
            .select(
                col("internal_id").alias("so_internal_id"),
                col("customer_internal_id").alias("so_customer_internal_id"),
                col("order_date").alias("so_order_date"),
                col("order_status").alias("so_order_status"),
                col("sales_channel").alias("so_sales_channel"),
                col("payment_status").alias("so_payment_status"),
                col("currency_code").alias("so_currency_code"),
                col("integration_source").alias("so_integration_source"),
                col("order_total").alias("so_order_total"),
                col("source_system").alias("so_source_system"),
                col("etl_run_id").alias("so_etl_run_id"),
                col("record_hash").alias("so_record_hash")
            )
        )

        df_sh = (
            spark.table(silver_shipments)
            .select(
                col("internal_id").alias("sh_internal_id"),
                col("sales_order_internal_id").alias("sh_sales_order_internal_id"),
                col("shipment_date").alias("sh_shipment_date"),
                col("delivery_date").alias("sh_delivery_date"),
                col("shipment_status").alias("sh_shipment_status"),
                col("warehouse_name").alias("sh_warehouse_name"),
                col("tracking_number").alias("sh_tracking_number"),
                col("record_hash").alias("sh_record_hash")
            )
        )

        # -------------------------------------------------
        # 2. Join transactional tables
        # -------------------------------------------------
        df_joined = (
            df_sol
            .join(
                df_so,
                col("sol_sales_order_internal_id") == col("so_internal_id"),
                "inner"
            )
            .join(
                df_sh,
                col("so_internal_id") == col("sh_sales_order_internal_id"),
                "left"
            )
        )

        rows_read = df_joined.count()

        # -------------------------------------------------
        # 3. Lookup dimensions
        # -------------------------------------------------
        df_customer_lkp = spark.table(gold_dim_customer).select(
            col("customer_key"),
            col("customer_internal_id").cast("int").alias("lkp_customer_internal_id")
        )

        df_item_lkp = spark.table(gold_dim_item).select(
            col("item_key"),
            col("item_internal_id").cast("int").alias("lkp_item_internal_id")
        )

        df_date_lkp = spark.table(gold_dim_date).select(
            col("date_key"),
            col("full_date").alias("lkp_full_date")
        )

        df_fact_pre = (
            df_joined
            .join(
                df_customer_lkp,
                col("so_customer_internal_id").cast("int") == col("lkp_customer_internal_id"),
                "left"
            )
            .join(
                df_item_lkp,
                col("sol_item_internal_id").cast("int") == col("lkp_item_internal_id"),
                "left"
            )
            .join(
                df_date_lkp,
                F.to_date(col("so_order_date")) == col("lkp_full_date"),
                "left"
            )
        )

        # -------------------------------------------------
        # 4. Reject dimension lookup failures
        # -------------------------------------------------
        reject_condition = (
            col("customer_key").isNull() |
            col("item_key").isNull() |
            col("date_key").isNull()
        )

        df_rejects = (
            df_fact_pre
            .filter(reject_condition)
            .withColumn("source_record_id", col("sol_internal_id").cast("string"))
            .withColumn("reject_stage", lit("GOLD"))
            .withColumn("reject_reason", lit("DIMENSION_LOOKUP_FAILED"))
            .withColumn("reject_column", lit("customer_key,item_key,order_date_key"))
            .withColumn("severity", lit("ERROR"))
            .withColumn("raw_payload", to_json(struct(*[col(c) for c in df_fact_pre.columns])))
            .withColumn("error_code", lit("GLD_001"))
            .withColumn("error_message", lit("One or more dimension lookups failed"))
        )

        rows_rejected = df_rejects.count()
        log_rejects(df_rejects, run_id, table_name)

        df_valid = df_fact_pre.filter(~reject_condition)

        # -------------------------------------------------
        # 5. Build fact output
        # -------------------------------------------------
        df_tgt = (
            df_valid
            .select(
                col("sol_internal_id").alias("sales_order_line_internal_id"),
                col("sol_sales_order_internal_id").alias("sales_order_internal_id"),
                col("sol_line_num").alias("sales_order_line_num"),
                col("date_key").alias("order_date_key"),
                col("customer_key"),
                col("item_key"),
                col("sol_quantity").alias("quantity"),
                col("sol_unit_rate").alias("unit_rate"),
                col("sol_line_amount").alias("line_amount"),
                col("so_order_total").alias("order_total"),
                col("so_order_status").alias("order_status"),
                col("so_sales_channel").alias("sales_channel"),
                col("so_payment_status").alias("payment_status"),
                col("so_currency_code").alias("currency_code"),
                col("so_integration_source").alias("integration_source"),
                col("sh_shipment_date").alias("shipment_date"),
                col("sh_delivery_date").alias("delivery_date"),
                col("sh_shipment_status").alias("shipment_status"),
                col("sh_warehouse_name").alias("warehouse_name"),
                col("sh_tracking_number").alias("tracking_number"),
                col("so_source_system").alias("source_system"),
                col("so_etl_run_id").alias("etl_run_id"),
                F.sha2(
                    F.concat_ws(
                        "||",
                        F.coalesce(col("sol_record_hash"), lit("")),
                        F.coalesce(col("so_record_hash"), lit("")),
                        F.coalesce(col("sh_record_hash"), lit(""))
                    ),
                    256
                ).alias("record_hash"),
                col("sol_created_at").alias("created_at"),
                current_timestamp().alias("etl_loaded_at")
            )
        )

        rows_written = df_tgt.count()
        print("df_tgt rows =", rows_written)

        # -------------------------------------------------
        # 6. Identity-safe load: TRUNCATE + INSERT INTO
        # -------------------------------------------------
        spark.sql(f"TRUNCATE TABLE {gold_fact_sales}")

        df_tgt.createOrReplaceTempView("vw_fact_sales_load")

        spark.sql(f"""
            INSERT INTO {gold_fact_sales} (
                sales_order_line_internal_id,
                sales_order_internal_id,
                sales_order_line_num,
                order_date_key,
                customer_key,
                item_key,
                quantity,
                unit_rate,
                line_amount,
                order_total,
                order_status,
                sales_channel,
                payment_status,
                currency_code,
                integration_source,
                shipment_date,
                delivery_date,
                shipment_status,
                warehouse_name,
                tracking_number,
                source_system,
                etl_run_id,
                record_hash,
                created_at,
                etl_loaded_at
            )
            SELECT
                sales_order_line_internal_id,
                sales_order_internal_id,
                sales_order_line_num,
                order_date_key,
                customer_key,
                item_key,
                quantity,
                unit_rate,
                line_amount,
                order_total,
                order_status,
                sales_channel,
                payment_status,
                currency_code,
                integration_source,
                shipment_date,
                delivery_date,
                shipment_status,
                warehouse_name,
                tracking_number,
                source_system,
                etl_run_id,
                record_hash,
                created_at,
                etl_loaded_at
            FROM vw_fact_sales_load
        """)

        records_after = spark.table(gold_fact_sales).count()

        end_audit_success(
            run_id=run_id,
            rows_read=rows_read,
            rows_written=rows_written,
            rows_rejected=rows_rejected,
            records_before=records_before,
            records_after=records_after
        )

        print(f"{gold_fact_sales}: {records_after} rows loaded successfully")

    except Exception as e:
        end_audit_fail(run_id, str(e))
        raise

load_fact_sales()

# Dimensions Full Load with file ingestion logs

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, lit, current_timestamp, to_json, struct,
    year, month, quarter, weekofyear, dayofmonth, date_format,
    dayofweek, when, explode, sequence
)
import uuid
import datetime

# =========================================================
# CONFIG
# =========================================================
pipeline_name = "retail_pipeline"
layer_name = "gold"
load_type = "FULL"
triggered_by = "manual"
source_system = "Silver_Tables"

audit_table = "retail.audit.etl_run_log"
reject_table = "retail.audit.etl_reject_log"
processed_runs_table = "retail.audit.layer_processed_runs"

batch_id = "batch_gold_dim_full_" + str(datetime.date.today())

# Source silver tables
silver_brand = "retail.silver.ns_brands"
silver_customer = "retail.silver.ns_customers"
silver_item = "retail.silver.ns_items"
silver_sales_orders = "retail.silver.ns_sales_orders"
silver_shipments = "retail.silver.ns_shipments"

# Target gold tables
gold_dim_brand = "retail.gold.dim_brand"
gold_dim_customer = "retail.gold.dim_customer"
gold_dim_item = "retail.gold.dim_item"
gold_dim_date = "retail.gold.dim_date"

# =========================================================
# HELPERS
# =========================================================
def generate_run_id():
    return str(uuid.uuid4())


def start_audit(run_id, table_name):
    spark.sql(f"""
        INSERT INTO {audit_table}
        (
          run_id, batch_id, pipeline_name, layer_name, table_name, source_system,
          load_type, triggered_by, start_time, end_time, duration_seconds,
          status, rows_read, rows_written, rows_rejected, records_before,
          records_after, error_message, created_at
        )
        VALUES (
          '{run_id}', '{batch_id}', '{pipeline_name}', '{layer_name}', '{table_name}', '{source_system}',
          '{load_type}', '{triggered_by}', current_timestamp(), NULL, NULL,
          'STARTED', NULL, NULL, NULL, NULL,
          NULL, NULL, current_timestamp()
        )
    """)


def end_audit_success(run_id, rows_read, rows_written, rows_rejected, records_before, records_after):
    spark.sql(f"""
        UPDATE {audit_table}
        SET
          end_time = current_timestamp(),
          duration_seconds = unix_timestamp(current_timestamp()) - unix_timestamp(start_time),
          status = 'SUCCESS',
          rows_read = {rows_read},
          rows_written = {rows_written},
          rows_rejected = {rows_rejected},
          records_before = {records_before},
          records_after = {records_after}
        WHERE run_id = '{run_id}'
    """)


def end_audit_fail(run_id, error_message):
    safe_error = error_message.replace("'", "''")[:5000]
    spark.sql(f"""
        UPDATE {audit_table}
        SET
          end_time = current_timestamp(),
          duration_seconds = unix_timestamp(current_timestamp()) - unix_timestamp(start_time),
          status = 'FAILED',
          error_message = '{safe_error}'
        WHERE run_id = '{run_id}'
    """)


def get_silver_run_ids_for_table(silver_table):
    return [
        r.etl_run_id
        for r in spark.table(silver_table)
        .select("etl_run_id")
        .where(col("etl_run_id").isNotNull())
        .distinct()
        .collect()
    ]


def log_layer_processed_runs_gold(
    source_run_ids,
    target_run_id,
    table_name,
    rows_read,
    rows_written,
    rows_rejected
):
    for source_run_id in source_run_ids:
        spark.sql(f"""
            INSERT INTO {processed_runs_table}
            SELECT
              uuid() AS tracking_id,
              '{source_run_id}' AS source_run_id,
              '{target_run_id}' AS target_run_id,
              '{pipeline_name}' AS pipeline_name,
              'silver' AS source_layer,
              'gold' AS target_layer,
              '{table_name}' AS table_name,
              'SUCCESS' AS status,
              CAST({rows_read} AS BIGINT) AS rows_read,
              CAST({rows_written} AS BIGINT) AS rows_written,
              CAST({rows_rejected} AS BIGINT) AS rows_rejected,
              current_timestamp() AS start_time,
              current_timestamp() AS end_time,
              0 AS duration_seconds,
              NULL AS error_message,
              current_timestamp() AS created_at,
              current_timestamp() AS updated_at
        """)


# =========================================================
# DIM BRAND
# =========================================================
def load_dim_brand():
    table_name = "dim_brand"
    run_id = generate_run_id()
    start_audit(run_id, table_name)

    try:
        df_src = spark.table(silver_brand)
        rows_read = df_src.count()
        records_before = spark.table(gold_dim_brand).count()

        df_tgt = (
            df_src
            .select(
                col("internal_id").alias("brand_internal_id"),
                "brand_code",
                "brand_name",
                "is_active",
                "source_system",
                "etl_run_id",
                "record_hash",
                "created_at",
                "etl_loaded_at"
            )
            .dropDuplicates(["brand_internal_id"])
        )

        rows_written = df_tgt.count()

        spark.sql(f"TRUNCATE TABLE {gold_dim_brand}")

        df_tgt.write.format("delta").mode("append").saveAsTable(gold_dim_brand)

        records_after = spark.table(gold_dim_brand).count()

        log_layer_processed_runs_gold(
            source_run_ids=get_silver_run_ids_for_table(silver_brand),
            target_run_id=run_id,
            table_name=table_name,
            rows_read=rows_read,
            rows_written=rows_written,
            rows_rejected=0
        )

        end_audit_success(run_id, rows_read, rows_written, 0, records_before, records_after)

        print(f"{gold_dim_brand}: {rows_written} rows loaded successfully")

    except Exception as e:
        end_audit_fail(run_id, str(e))
        raise


# =========================================================
# DIM CUSTOMER
# =========================================================
def load_dim_customer():
    table_name = "dim_customer"
    run_id = generate_run_id()
    start_audit(run_id, table_name)

    try:
        df_src = spark.table(silver_customer)
        rows_read = df_src.count()
        records_before = spark.table(gold_dim_customer).count()

        df_tgt = (
            df_src
            .select(
                col("internal_id").alias("customer_internal_id"),
                "entity_id",
                "company_name",
                "customer_type",
                "email",
                "phone",
                "subsidiary",
                "currency_code",
                "terms_name",
                "vat_number",
                "billing_address1",
                "billing_city",
                "billing_state",
                "billing_country",
                "shipping_address1",
                "shipping_city",
                "shipping_state",
                "shipping_country",
                "is_active",
                "source_system",
                "etl_run_id",
                "record_hash",
                "created_at",
                "etl_loaded_at"
            )
            .dropDuplicates(["customer_internal_id"])
        )

        rows_written = df_tgt.count()

        spark.sql(f"TRUNCATE TABLE {gold_dim_customer}")

        df_tgt.write.format("delta").mode("append").saveAsTable(gold_dim_customer)

        records_after = spark.table(gold_dim_customer).count()

        log_layer_processed_runs_gold(
            source_run_ids=get_silver_run_ids_for_table(silver_customer),
            target_run_id=run_id,
            table_name=table_name,
            rows_read=rows_read,
            rows_written=rows_written,
            rows_rejected=0
        )

        end_audit_success(run_id, rows_read, rows_written, 0, records_before, records_after)

        print(f"{gold_dim_customer}: {rows_written} rows loaded successfully")

    except Exception as e:
        end_audit_fail(run_id, str(e))
        raise


# =========================================================
# DIM ITEM
# =========================================================
def load_dim_item():
    table_name = "dim_item"
    run_id = generate_run_id()
    start_audit(run_id, table_name)

    try:
        df_src = spark.table(silver_item)
        rows_read = df_src.count()
        records_before = spark.table(gold_dim_item).count()

        df_tgt = (
            df_src
            .select(
                col("internal_id").alias("item_internal_id"),
                "item_id",
                "item_name",
                "item_type",
                "category",
                "subcategory",
                "brand_internal_id",
                "base_price",
                "cost_price",
                "is_active",
                "source_system",
                "etl_run_id",
                "record_hash",
                "created_at",
                "etl_loaded_at"
            )
            .dropDuplicates(["item_internal_id"])
        )

        rows_written = df_tgt.count()

        spark.sql(f"TRUNCATE TABLE {gold_dim_item}")

        df_tgt.write.format("delta").mode("append").saveAsTable(gold_dim_item)

        records_after = spark.table(gold_dim_item).count()

        log_layer_processed_runs_gold(
            source_run_ids=get_silver_run_ids_for_table(silver_item),
            target_run_id=run_id,
            table_name=table_name,
            rows_read=rows_read,
            rows_written=rows_written,
            rows_rejected=0
        )

        end_audit_success(run_id, rows_read, rows_written, 0, records_before, records_after)

        print(f"{gold_dim_item}: {rows_written} rows loaded successfully")

    except Exception as e:
        end_audit_fail(run_id, str(e))
        raise


# =========================================================
# DIM DATE
# =========================================================
def load_dim_date():
    table_name = "dim_date"
    run_id = generate_run_id()
    start_audit(run_id, table_name)

    try:
        df_orders = spark.table(silver_sales_orders).select(col("order_date").alias("dt"))
        df_shipments_1 = spark.table(silver_shipments).select(col("shipment_date").alias("dt"))
        df_shipments_2 = spark.table(silver_shipments).select(col("delivery_date").alias("dt"))

        df_dates = (
            df_orders.union(df_shipments_1).union(df_shipments_2)
            .filter(col("dt").isNotNull())
        )

        rows_read = df_dates.count()
        records_before = spark.table(gold_dim_date).count()

        min_max = df_dates.agg(
            F.min("dt").alias("min_date"),
            F.max("dt").alias("max_date")
        ).collect()[0]

        min_date = min_max["min_date"]
        max_date = min_max["max_date"]

        df_calendar = (
            spark.createDataFrame([(min_date, max_date)], ["start_date", "end_date"])
            .select(explode(sequence(col("start_date"), col("end_date"))).alias("full_date"))
        )

        df_tgt = (
            df_calendar
            .select(
                date_format(col("full_date"), "yyyyMMdd").cast("int").alias("date_key"),
                col("full_date"),
                dayofmonth(col("full_date")).alias("day_of_month"),
                month(col("full_date")).alias("month_num"),
                date_format(col("full_date"), "MMMM").alias("month_name"),
                quarter(col("full_date")).alias("quarter_num"),
                year(col("full_date")).alias("year_num"),
                weekofyear(col("full_date")).alias("week_of_year"),
                dayofweek(col("full_date")).alias("day_of_week_num"),
                date_format(col("full_date"), "EEEE").alias("day_of_week_name"),
                when(dayofweek(col("full_date")).isin(1, 7), True).otherwise(False).alias("is_weekend")
            )
        )

        rows_written = df_tgt.count()

        spark.sql(f"TRUNCATE TABLE {gold_dim_date}")

        df_tgt.write.format("delta").mode("append").saveAsTable(gold_dim_date)

        records_after = spark.table(gold_dim_date).count()

        source_run_ids = list(set(
            get_silver_run_ids_for_table(silver_sales_orders) +
            get_silver_run_ids_for_table(silver_shipments)
        ))

        log_layer_processed_runs_gold(
            source_run_ids=source_run_ids,
            target_run_id=run_id,
            table_name=table_name,
            rows_read=rows_read,
            rows_written=rows_written,
            rows_rejected=0
        )

        end_audit_success(run_id, rows_read, rows_written, 0, records_before, records_after)

        print(f"{gold_dim_date}: {rows_written} rows loaded successfully")

    except Exception as e:
        end_audit_fail(run_id, str(e))
        raise


# =========================================================
# RUN DIMENSIONS
# =========================================================
load_dim_brand()
load_dim_customer()
load_dim_item()
load_dim_date()

# Gold Fact Full Load Script

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, lit, current_timestamp, to_json, struct
)
import uuid
import datetime

# =========================================================
# CONFIG
# =========================================================
pipeline_name = "retail_pipeline"
layer_name = "gold"
load_type = "FULL"
triggered_by = "manual"
source_system = "Silver_Tables"

audit_table = "retail.audit.etl_run_log"
reject_table = "retail.audit.etl_reject_log"
processed_runs_table = "retail.audit.layer_processed_runs"

batch_id = "batch_gold_fact_full_" + str(datetime.date.today())

# Source silver tables
silver_sales_orders = "retail.silver.ns_sales_orders"
silver_sales_order_lines = "retail.silver.ns_sales_order_lines"
silver_shipments = "retail.silver.ns_shipments"

# Target gold tables
gold_dim_customer = "retail.gold.dim_customer"
gold_dim_item = "retail.gold.dim_item"
gold_dim_date = "retail.gold.dim_date"
gold_fact_sales = "retail.gold.fact_sales"

# =========================================================
# HELPERS
# =========================================================
def generate_run_id():
    return str(uuid.uuid4())


def start_audit(run_id, table_name):
    spark.sql(f"""
        INSERT INTO {audit_table}
        (
          run_id, batch_id, pipeline_name, layer_name, table_name, source_system,
          load_type, triggered_by, start_time, end_time, duration_seconds,
          status, rows_read, rows_written, rows_rejected, records_before,
          records_after, error_message, created_at
        )
        VALUES (
          '{run_id}', '{batch_id}', '{pipeline_name}', '{layer_name}', '{table_name}', '{source_system}',
          '{load_type}', '{triggered_by}', current_timestamp(), NULL, NULL,
          'STARTED', NULL, NULL, NULL, NULL,
          NULL, NULL, current_timestamp()
        )
    """)


def end_audit_success(run_id, rows_read, rows_written, rows_rejected, records_before, records_after):
    spark.sql(f"""
        UPDATE {audit_table}
        SET
          end_time = current_timestamp(),
          duration_seconds = unix_timestamp(current_timestamp()) - unix_timestamp(start_time),
          status = 'SUCCESS',
          rows_read = {rows_read},
          rows_written = {rows_written},
          rows_rejected = {rows_rejected},
          records_before = {records_before},
          records_after = {records_after}
        WHERE run_id = '{run_id}'
    """)


def end_audit_fail(run_id, error_message):
    safe_error = error_message.replace("'", "''")[:5000]
    spark.sql(f"""
        UPDATE {audit_table}
        SET
          end_time = current_timestamp(),
          duration_seconds = unix_timestamp(current_timestamp()) - unix_timestamp(start_time),
          status = 'FAILED',
          error_message = '{safe_error}'
        WHERE run_id = '{run_id}'
    """)


def log_rejects(df_rejects, run_id, table_name):
    if df_rejects.isEmpty():
        return

    (
        df_rejects
        .withColumn("reject_id", F.expr("uuid()"))
        .withColumn("run_id", lit(run_id))
        .withColumn("batch_id", lit(batch_id))
        .withColumn("pipeline_name", lit(pipeline_name))
        .withColumn("layer_name", lit(layer_name))
        .withColumn("table_name", lit(table_name))
        .withColumn("source_system", lit(source_system))
        .withColumn("rejected_at", current_timestamp())
        .withColumn("created_at", current_timestamp())
        .select(
            "reject_id", "run_id", "batch_id", "pipeline_name", "layer_name",
            "table_name", "source_system", "source_record_id", "reject_stage",
            "reject_reason", "reject_column", "severity", "raw_payload",
            "error_code", "error_message", "rejected_at", "created_at"
        )
        .write.format("delta").mode("append").saveAsTable(reject_table)
    )


def get_silver_run_ids_for_table(silver_table):
    return [
        r.etl_run_id
        for r in spark.table(silver_table)
        .select("etl_run_id")
        .where(col("etl_run_id").isNotNull())
        .distinct()
        .collect()
    ]


def log_layer_processed_runs_gold(
    source_run_ids,
    target_run_id,
    table_name,
    rows_read,
    rows_written,
    rows_rejected
):
    for source_run_id in source_run_ids:
        spark.sql(f"""
            INSERT INTO {processed_runs_table}
            SELECT
              uuid() AS tracking_id,
              '{source_run_id}' AS source_run_id,
              '{target_run_id}' AS target_run_id,
              '{pipeline_name}' AS pipeline_name,
              'silver' AS source_layer,
              'gold' AS target_layer,
              '{table_name}' AS table_name,
              'SUCCESS' AS status,
              CAST({rows_read} AS BIGINT) AS rows_read,
              CAST({rows_written} AS BIGINT) AS rows_written,
              CAST({rows_rejected} AS BIGINT) AS rows_rejected,
              current_timestamp() AS start_time,
              current_timestamp() AS end_time,
              0 AS duration_seconds,
              NULL AS error_message,
              current_timestamp() AS created_at,
              current_timestamp() AS updated_at
        """)


# =========================================================
# FACT SALES
# =========================================================
def load_fact_sales():
    table_name = "fact_sales"
    run_id = generate_run_id()
    start_audit(run_id, table_name)

    try:
        records_before = spark.table(gold_fact_sales).count()

        df_sol = (
            spark.table(silver_sales_order_lines)
            .select(
                col("internal_id").alias("sol_internal_id"),
                col("sales_order_internal_id").alias("sol_sales_order_internal_id"),
                col("line_num").alias("sol_line_num"),
                col("item_internal_id").alias("sol_item_internal_id"),
                col("quantity").alias("sol_quantity"),
                col("unit_rate").alias("sol_unit_rate"),
                col("line_amount").alias("sol_line_amount"),
                col("record_hash").alias("sol_record_hash"),
                col("created_at").alias("sol_created_at")
            )
        )

        df_so = (
            spark.table(silver_sales_orders)
            .select(
                col("internal_id").alias("so_internal_id"),
                col("customer_internal_id").alias("so_customer_internal_id"),
                col("order_date").alias("so_order_date"),
                col("order_status").alias("so_order_status"),
                col("sales_channel").alias("so_sales_channel"),
                col("payment_status").alias("so_payment_status"),
                col("currency_code").alias("so_currency_code"),
                col("integration_source").alias("so_integration_source"),
                col("order_total").alias("so_order_total"),
                col("source_system").alias("so_source_system"),
                col("etl_run_id").alias("so_etl_run_id"),
                col("record_hash").alias("so_record_hash")
            )
        )

        df_sh = (
            spark.table(silver_shipments)
            .select(
                col("internal_id").alias("sh_internal_id"),
                col("sales_order_internal_id").alias("sh_sales_order_internal_id"),
                col("shipment_date").alias("sh_shipment_date"),
                col("delivery_date").alias("sh_delivery_date"),
                col("shipment_status").alias("sh_shipment_status"),
                col("warehouse_name").alias("sh_warehouse_name"),
                col("tracking_number").alias("sh_tracking_number"),
                col("record_hash").alias("sh_record_hash")
            )
        )

        df_joined = (
            df_sol
            .join(
                df_so,
                col("sol_sales_order_internal_id") == col("so_internal_id"),
                "inner"
            )
            .join(
                df_sh,
                col("so_internal_id") == col("sh_sales_order_internal_id"),
                "left"
            )
        )

        rows_read = df_joined.count()

        df_customer_lkp = spark.table(gold_dim_customer).select(
            col("customer_key"),
            col("customer_internal_id").cast("int").alias("lkp_customer_internal_id")
        )

        df_item_lkp = spark.table(gold_dim_item).select(
            col("item_key"),
            col("item_internal_id").cast("int").alias("lkp_item_internal_id")
        )

        df_date_lkp = spark.table(gold_dim_date).select(
            col("date_key"),
            col("full_date").alias("lkp_full_date")
        )

        df_fact_pre = (
            df_joined
            .join(
                df_customer_lkp,
                col("so_customer_internal_id").cast("int") == col("lkp_customer_internal_id"),
                "left"
            )
            .join(
                df_item_lkp,
                col("sol_item_internal_id").cast("int") == col("lkp_item_internal_id"),
                "left"
            )
            .join(
                df_date_lkp,
                F.to_date(col("so_order_date")) == col("lkp_full_date"),
                "left"
            )
        )

        reject_condition = (
            col("customer_key").isNull() |
            col("item_key").isNull() |
            col("date_key").isNull()
        )

        df_rejects = (
            df_fact_pre
            .filter(reject_condition)
            .withColumn("source_record_id", col("sol_internal_id").cast("string"))
            .withColumn("reject_stage", lit("GOLD"))
            .withColumn("reject_reason", lit("DIMENSION_LOOKUP_FAILED"))
            .withColumn("reject_column", lit("customer_key,item_key,order_date_key"))
            .withColumn("severity", lit("ERROR"))
            .withColumn("raw_payload", to_json(struct(*[col(c) for c in df_fact_pre.columns])))
            .withColumn("error_code", lit("GLD_001"))
            .withColumn("error_message", lit("One or more dimension lookups failed"))
        )

        rows_rejected = df_rejects.count()
        log_rejects(df_rejects, run_id, table_name)

        df_valid = df_fact_pre.filter(~reject_condition)

        df_tgt = (
            df_valid
            .select(
                col("sol_internal_id").alias("sales_order_line_internal_id"),
                col("sol_sales_order_internal_id").alias("sales_order_internal_id"),
                col("sol_line_num").alias("sales_order_line_num"),
                col("date_key").alias("order_date_key"),
                col("customer_key"),
                col("item_key"),
                col("sol_quantity").alias("quantity"),
                col("sol_unit_rate").alias("unit_rate"),
                col("sol_line_amount").alias("line_amount"),
                col("so_order_total").alias("order_total"),
                col("so_order_status").alias("order_status"),
                col("so_sales_channel").alias("sales_channel"),
                col("so_payment_status").alias("payment_status"),
                col("so_currency_code").alias("currency_code"),
                col("so_integration_source").alias("integration_source"),
                col("sh_shipment_date").alias("shipment_date"),
                col("sh_delivery_date").alias("delivery_date"),
                col("sh_shipment_status").alias("shipment_status"),
                col("sh_warehouse_name").alias("warehouse_name"),
                col("sh_tracking_number").alias("tracking_number"),
                col("so_source_system").alias("source_system"),
                col("so_etl_run_id").alias("etl_run_id"),
                F.sha2(
                    F.concat_ws(
                        "||",
                        F.coalesce(col("sol_record_hash"), lit("")),
                        F.coalesce(col("so_record_hash"), lit("")),
                        F.coalesce(col("sh_record_hash"), lit(""))
                    ),
                    256
                ).alias("record_hash"),
                col("sol_created_at").alias("created_at"),
                current_timestamp().alias("etl_loaded_at")
            )
        )

        rows_written = df_tgt.count()

        spark.sql(f"TRUNCATE TABLE {gold_fact_sales}")

        df_tgt.createOrReplaceTempView("vw_fact_sales_load")

        spark.sql(f"""
            INSERT INTO {gold_fact_sales} (
                sales_order_line_internal_id,
                sales_order_internal_id,
                sales_order_line_num,
                order_date_key,
                customer_key,
                item_key,
                quantity,
                unit_rate,
                line_amount,
                order_total,
                order_status,
                sales_channel,
                payment_status,
                currency_code,
                integration_source,
                shipment_date,
                delivery_date,
                shipment_status,
                warehouse_name,
                tracking_number,
                source_system,
                etl_run_id,
                record_hash,
                created_at,
                etl_loaded_at
            )
            SELECT
                sales_order_line_internal_id,
                sales_order_internal_id,
                sales_order_line_num,
                order_date_key,
                customer_key,
                item_key,
                quantity,
                unit_rate,
                line_amount,
                order_total,
                order_status,
                sales_channel,
                payment_status,
                currency_code,
                integration_source,
                shipment_date,
                delivery_date,
                shipment_status,
                warehouse_name,
                tracking_number,
                source_system,
                etl_run_id,
                record_hash,
                created_at,
                etl_loaded_at
            FROM vw_fact_sales_load
        """)

        records_after = spark.table(gold_fact_sales).count()

        source_run_ids = list(set(
            get_silver_run_ids_for_table(silver_sales_order_lines) +
            get_silver_run_ids_for_table(silver_sales_orders) +
            get_silver_run_ids_for_table(silver_shipments)
        ))

        log_layer_processed_runs_gold(
            source_run_ids=source_run_ids,
            target_run_id=run_id,
            table_name=table_name,
            rows_read=rows_read,
            rows_written=rows_written,
            rows_rejected=rows_rejected
        )

        end_audit_success(
            run_id=run_id,
            rows_read=rows_read,
            rows_written=rows_written,
            rows_rejected=rows_rejected,
            records_before=records_before,
            records_after=records_after
        )

        print(f"{gold_fact_sales}: {records_after} rows loaded successfully")

    except Exception as e:
        end_audit_fail(run_id, str(e))
        raise


# =========================================================
# RUN FACT
# =========================================================
load_fact_sales()